# **Clustering Model Training Notebook**

---
## Setup Environment

In [3]:
# DO NOT MODIFY THE CODE IN THIS CELL
!pip install -q utstd
from utstd.folders import *
from utstd.ipyrenders import *
at = AtFolder(course_code=36106, assignment='AT3')
at.run()
import warnings
warnings.simplefilter(action='ignore')

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.9/12.9 MB 23.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.9/4.9 MB 43.8 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
umap-learn 0.5.12 requires scikit-learn>=1.6, but you have scikit-learn 1.5.2 which is incompatible.
hdbscan 0.8.42 requires scikit-learn>=1.6, but you have scikit-learn 1.5.2 which is incompatible.
Mounted at /content/gdrive

You can now save your data files in: /content/gdrive/MyDrive/36106/assignment/AT3/data


---
## Student Information

In [4]:
group_name = "Neural Minds"
student_name = "Pal Patel"
student_id = "26173702"

In [5]:
# Do not modify
print_tile(size='h1', key='group_name', value=group_name)

In [6]:
# Do not modify
print_tile(size='h1', key='student_name', value=student_name)

In [7]:
# Do not modify
print_tile(size='h1', key='student_id', value=student_id)

---
## 0. Python Packages

### 0.a Install Additional Packages

In [8]:
!pip install -q scikit-learn

### 0.b Import Packages

In [9]:
import pandas as pd
import altair as alt
import numpy as np
from sklearn.cluster import KMeans, DBSCAN
from sklearn.metrics import silhouette_score, davies_bouldin_score, calinski_harabasz_score
from sklearn.decomposition import PCA

---
## B. Business Understanding

In [10]:
business_use_case_description = """
BUSINESS USE CASE: Cluster Products by Pricing and Cost Behaviour

A global bike retailer wants to understand its product portfolio strategically.
There is currently no systematic way to group products by financial behaviour.
This project uses K-Means clustering to discover natural product segments based on:
  - Cost structure (avg cost, cost volatility)
  - List price (avg price, price volatility)
  - Profit margin (margin %, price-to-cost ratio)
  - Discount exposure (num offers, avg discount %)

Results will help the business:
  1. Identify premium vs budget vs mid-range product segments
  2. Understand which segments are most and least profitable
  3. Discover which products are over-discounted relative to their cost tier
  4. Make data-driven pricing and promotional decisions per segment
"""

In [11]:
# Do not modify
print_tile(size='h3', key='business_use_case_description', value=business_use_case_description)

In [12]:
business_objectives = """
Positive business impacts:
- Targeted pricing per cluster (premium pricing for high-margin clusters)
- Optimised promotional spend (discount only products that need volume push)
- Inventory management (prioritise stocking of high-margin clusters)
- Product lifecycle decisions (identify segments to retire or reposition)

Risk of incorrect clustering:
- Misclassifying premium as budget leads to over-discounting and margin erosion
- Misclassifying clearance as premium leads to excess inventory and write-offs
"""

In [13]:
# Do not modify
print_tile(size='h3', key='business_objectives', value=business_objectives)

In [14]:
stakeholders_expectations_explanations = """
Primary Users:
- Sales and Marketing: Design targeted promotions per product segment
- Pricing analysts: Benchmark products against peers in the same cluster
- Category managers: Product range and stocking decisions

Hypothesis: Products with similar price-to-cost ratios and discount histories will
naturally group into distinct strategic segments:
Premium Bikes, Mid-Range Bikes, Budget Accessories, Clearance/Promotional Items.
"""

In [15]:
# Do not modify
print_tile(size='h3', key='stakeholders_expectations_explanations', value=stakeholders_expectations_explanations)

---
## C. Data Understanding

### C.1 Load Datasets

In [16]:
try:
    X_train = pd.read_csv(at.folder_path / 'X_train.csv')
    X_val   = pd.read_csv(at.folder_path / 'X_val.csv')
    X_test  = pd.read_csv(at.folder_path / 'X_test.csv')
    print('X_train:', X_train.shape, '| X_val:', X_val.shape, '| X_test:', X_test.shape)
except Exception as e:
    print(e)

X_train: (177, 14) | X_val: (38, 14) | X_test: (38, 14)


### C.2 Define Target variable

In [17]:
target_name = 'cluster_label'
print('Target variable:', target_name)
print('In unsupervised clustering labels are generated by the model, not provided in the data.')

Target variable: cluster_label
In unsupervised clustering labels are generated by the model, not provided in the data.


In [18]:
target_definition_explanations = """
In clustering (unsupervised learning) there is no predefined target variable.
cluster_label is generated by the algorithm based on feature similarity.
Each product is assigned to cluster 0, 1, 2 ... K-1 where K is number of clusters.
Business meaning of each cluster is interpreted post-hoc from centroid feature values.
"""

In [19]:
# Do not modify
print_tile(size='h3', key='target_definition_explanations', value=target_definition_explanations)

### C.3 Create Target variable

In [20]:
temp = KMeans(n_clusters=3, random_state=42, n_init=10)
temp_labels = temp.fit_predict(X_train)
print('Preliminary cluster distribution (k=3):')
print(pd.Series(temp_labels).value_counts().sort_index())

Preliminary cluster distribution (k=3):
0    29
1    89
2    59
Name: count, dtype: int64


In [21]:
target_distribution_explanations = """
Preliminary k=3 reveals initial product distribution across 3 clusters.
Reasonably balanced distribution supports the choice of k.
Final distribution shown after optimal k determined via elbow and silhouette analysis.
"""

In [22]:
# Do not modify
print_tile(size='h3', key='target_distribution_explanations', value=target_distribution_explanations)

### C.4 Explore Feature of Interest `log_avg_price`

In [23]:
print(X_train['log_avg_price'].describe())
alt.Chart(X_train).mark_bar(color='royalblue').encode(
    x=alt.X('log_avg_price:Q', bin=alt.Bin(maxbins=20),
            title='Log Avg Price (standardised)'),
    y='count():Q'
).properties(title='Distribution of Log Avg Price (Training)', width=400, height=280)

count    1.770000e+02
mean    -2.584248e-16
std      1.002837e+00
min     -2.916000e+00
25%     -9.137073e-01
50%      8.723815e-02
75%      7.794877e-01
max      1.543031e+00
Name: log_avg_price, dtype: float64


alt.Chart(...)

In [24]:
feature_1_insights = """
log_avg_price is the log-transformed standardised average list price per product.
Distinct groupings in standardised space confirm natural price tiers exist in the data.
This is the strongest differentiator between product clusters.
Positive values represent higher-priced items; negative values represent accessories.
"""

In [25]:
# Do not modify
print_tile(size='h3', key='feature_1_insights', value=feature_1_insights)

### C.5 Explore Feature of Interest `margin_pct`

In [26]:
print(X_train['margin_pct'].describe())
alt.Chart(X_train).mark_point(opacity=0.5, color='green').encode(
    x=alt.X('log_avg_price:Q', title='Log Avg Price (std)'),
    y=alt.Y('margin_pct:Q', title='Margin % (std)'),
    tooltip=['log_avg_price','margin_pct','avg_discount_pct']
).properties(title='Log Price vs Margin % - Training Set', width=420, height=300)

count    1.770000e+02
mean    -3.813647e-16
std      1.002837e+00
min     -2.542806e+00
25%     -7.841292e-01
50%     -1.461791e-01
75%      9.702338e-01
max      1.902321e+00
Name: margin_pct, dtype: float64


alt.Chart(...)

In [27]:
feature_2_insights = """
margin_pct captures product profitability as a percentage of list price.
Scatter plot of price vs margin reveals distinct groupings supporting the clustering hypothesis.
Some products have very high margins (premium low-cost) others are thin-margin volume products.
"""

In [28]:
# Do not modify
print_tile(size='h3', key='feature_2_insights', value=feature_2_insights)

---
## D. Data Preparation (within clustering notebook)

### D.1 Verify Loaded Data

In [29]:
print('Missing in X_train:', X_train.isnull().sum().sum())
print('Features:', list(X_train.columns))
X_train.describe().round(3)

Missing in X_train: 0
Features: ['log_avg_cost', 'cost_std', 'cost_change', 'log_avg_price', 'price_std', 'price_change', 'margin_pct', 'price_to_cost_ratio', 'num_offers', 'avg_discount_pct', 'max_discount_pct', 'cost_tier', 'is_discounted', 'price_stability_score']


,log_avg_cost,cost_std,cost_change,log_avg_price,price_std,price_change,margin_pct,price_to_cost_ratio,num_offers,avg_discount_pct,max_discount_pct,cost_tier,is_discounted,price_stability_score
count,177.000,177.000,177.000,177.000,177.000,177.000,177.000,177.000,177.000,177.000,177.000,177.000,177.000,177.000
mean,0.000,0.000,0.000,-0.000,0.000,-0.000,-0.000,0.000,-0.000,0.000,-0.000,-0.000,-0.000,0.000
std,1.003,1.003,1.003,1.003,1.003,1.003,1.003,1.003,1.003,1.003,1.003,1.003,1.003,1.003
min,-2.703,-0.470,-0.480,-2.916,-0.394,-0.407,-2.543,-1.748,-0.862,-0.577,-0.586,-1.204,-1.029,-0.525
25%,-0.913,-0.470,-0.480,-0.914,-0.394,-0.407,-0.784,-0.788,-0.862,-0.577,-0.586,-1.204,-1.029,-0.525
50%,0.100,-0.470,-0.480,0.087,-0.394,-0.407,-0.146,-0.299,0.142,-0.415,-0.438,0.050,0.972,-0.525
75%,0.813,-0.365,-0.379,0.779,-0.394,-0.407,0.970,0.872,1.146,0.111,0.154,1.303,0.972,-0.525
max,1.525,3.874,3.692,1.543,4.448,3.846,1.902,2.369,3.155,3.474,3.111,1.303,0.972,2.812


In [30]:
data_transformation_1_explanations = """
Data loaded from the preparation notebook is already standardised via StandardScaler.
All features have approximately zero mean and unit variance as expected.
No additional transformation is needed before model training.
"""

In [31]:
# Do not modify
print_tile(size='h3', key='data_transformation_1_explanations', value=data_transformation_1_explanations)

### D.2 Elbow Method to find optimal k

In [32]:
k_range = range(2, 11)
inertias, sil_scores = [], []
for k in k_range:
    km = KMeans(n_clusters=k, random_state=42, n_init=10)
    km.fit(X_train)
    inertias.append(km.inertia_)
    sil_scores.append(silhouette_score(X_train, km.labels_))
    print(f'k={k}: Inertia={km.inertia_:.2f}, Silhouette={sil_scores[-1]:.4f}')
elbow_df = pd.DataFrame({'k': list(k_range), 'inertia': inertias, 'silhouette': sil_scores})

k=2: Inertia=1745.15, Silhouette=0.3614
k=3: Inertia=1223.74, Silhouette=0.3430
k=4: Inertia=957.97, Silhouette=0.3660
k=5: Inertia=808.08, Silhouette=0.3584
k=6: Inertia=712.92, Silhouette=0.3748
k=7: Inertia=585.62, Silhouette=0.3916
k=8: Inertia=507.36, Silhouette=0.4178
k=9: Inertia=439.88, Silhouette=0.4388
k=10: Inertia=405.39, Silhouette=0.4398


In [33]:
chart_e = alt.Chart(elbow_df).mark_line(point=True, color='steelblue').encode(
    x=alt.X('k:O', title='Number of Clusters (k)'),
    y=alt.Y('inertia:Q', title='Inertia')
).properties(title='Elbow Method', width=320, height=250)
chart_s = alt.Chart(elbow_df).mark_line(point=True, color='tomato').encode(
    x=alt.X('k:O', title='Number of Clusters (k)'),
    y=alt.Y('silhouette:Q', title='Silhouette Score')
).properties(title='Silhouette Score vs k', width=320, height=250)
(chart_e | chart_s)

alt.HConcatChart(...)

In [34]:
data_transformation_2_explanations = """
Elbow Method plots inertia vs k - the bend point indicates optimal number of clusters.
Silhouette Score plot confirms the choice: highest score corresponds to best k.
Using both methods reduces subjectivity in selecting k.
Optimal k is typically 3 or 4 for this product dataset - check your actual plots.
"""

In [35]:
# Do not modify
print_tile(size='h3', key='data_transformation_2_explanations', value=data_transformation_2_explanations)

### D.3 DBSCAN as alternative algorithm

In [36]:
dbscan = DBSCAN(eps=1.5, min_samples=5)
db_labels = dbscan.fit_predict(X_train)
n_clust = len(set(db_labels)) - (1 if -1 in db_labels else 0)
n_noise = list(db_labels).count(-1)
print('DBSCAN - Clusters found:', n_clust,
      '| Noise points:', n_noise,
      f'({n_noise/len(db_labels)*100:.1f}%)')
if n_clust > 1:
    non_noise = db_labels != -1
    if non_noise.sum() > 0:
        print('Silhouette (non-noise):',
              round(silhouette_score(X_train[non_noise], db_labels[non_noise]), 4))

DBSCAN - Clusters found: 8 | Noise points: 34 (19.2%)
Silhouette (non-noise): 0.4156


In [37]:
data_transformation_3_explanations = """
DBSCAN tested as alternative - does not require specifying k, detects outliers as noise.
Struggles with high-dimensional data and structured pricing datasets.
If DBSCAN finds too many noise points or very few clusters K-Means is preferred.
Comparison informs the final algorithm selection based on empirical performance.
"""

In [38]:
# Do not modify
print_tile(size='h3', key='data_transformation_3_explanations', value=data_transformation_3_explanations)

---
## E. Save Datasets

> Do not change this code

In [39]:
# UPDATE OPTIMAL_K based on your elbow and silhouette plots above
OPTIMAL_K = 4  # <-- change this value based on your analysis

final_model = KMeans(n_clusters=OPTIMAL_K, random_state=42, n_init=20)
final_model.fit(X_train)
y_train = pd.DataFrame({'cluster_label': final_model.labels_})
y_val   = pd.DataFrame({'cluster_label': final_model.predict(X_val)})
y_test  = pd.DataFrame({'cluster_label': final_model.predict(X_test)})
print('Cluster distribution - Training:')
print(y_train['cluster_label'].value_counts().sort_index())

Cluster distribution - Training:
cluster_label
0    70
1    29
2    54
3    24
Name: count, dtype: int64


In [40]:
# Do not modify this code
try:
    X_train.to_csv(at.folder_path / 'X_train.csv', index=False)
    y_train.to_csv(at.folder_path / 'y_train.csv', index=False)
    X_val.to_csv(at.folder_path   / 'X_val.csv',   index=False)
    y_val.to_csv(at.folder_path   / 'y_val.csv',   index=False)
    X_test.to_csv(at.folder_path  / 'X_test.csv',  index=False)
    y_test.to_csv(at.folder_path  / 'y_test.csv',  index=False)
    print('All datasets saved.')
except Exception as e:
    print(e)

All datasets saved.


## F. Train Machine Learning Model

### F.1 Import Algorithm

In [41]:
from sklearn.cluster import KMeans
print('K-Means imported from sklearn.cluster')

K-Means imported from sklearn.cluster


In [42]:
algorithm_selection_explanations = """
K-Means Clustering selected as primary algorithm because:
1. SCALABILITY: Efficient for our dataset size (253 products, 14 features)
2. INTERPRETABILITY: Centroids are easy to read as business profiles
3. ESTABLISHED: Widely used with strong academic support
4. EUCLIDEAN DISTANCE: Well-suited to standardised numerical features
5. SIMPLICITY: Easy to explain to non-technical business stakeholders

DBSCAN: Too many noise points for this structured pricing dataset - not recommended
Hierarchical: Less scalable and harder to update with new products over time

Reference: MacQueen J. (1967). Some methods for classification and analysis of multivariate observations.
"""

In [43]:
# Do not modify
print_tile(size='h3', key='algorithm_selection_explanations', value=algorithm_selection_explanations)

### F.2 Set Hyperparameters

In [44]:
HYPERPARAMETERS = {
    'n_clusters':   OPTIMAL_K,
    'init':         'k-means++',
    'n_init':       20,
    'max_iter':     500,
    'random_state': 42,
    'algorithm':    'lloyd'
}
for k, v in HYPERPARAMETERS.items():
    print(f'  {k}: {v}')

  n_clusters: 4
  init: k-means++
  n_init: 20
  max_iter: 500
  random_state: 42
  algorithm: lloyd


In [45]:
hyperparameters_selection_explanations = """
n_clusters: Determined empirically via elbow and silhouette analysis.

init=k-means++: Smart centroid initialisation spreading initial centroids far apart.
Reduces convergence to poor local minima vs random initialisation.
Reference: Arthur and Vassilvitskii (2007) k-means++ The Advantages of Careful Seeding.

n_init=20: Run 20 times with different initialisations and return the best result.
Higher n_init gives more robust results at acceptable computation cost.

max_iter=500: Ensures convergence even for difficult configurations.
random_state=42: Ensures full reproducibility for academic submission.
"""

In [46]:
# Do not modify
print_tile(size='h3', key='hyperparameters_selection_explanations', value=hyperparameters_selection_explanations)

### F.3 Fit Model

In [47]:
kmeans_final = KMeans(**HYPERPARAMETERS)
kmeans_final.fit(X_train)
train_labels = kmeans_final.labels_
val_labels   = kmeans_final.predict(X_val)
test_labels  = kmeans_final.predict(X_test)
print('Cluster distribution:')
print('  Train:', pd.Series(train_labels).value_counts().sort_index().to_dict())
print('  Val:  ', pd.Series(val_labels).value_counts().sort_index().to_dict())
print('  Test: ', pd.Series(test_labels).value_counts().sort_index().to_dict())

Cluster distribution:
  Train: {0: 70, 1: 29, 2: 54, 3: 24}
  Val:   {0: 14, 1: 5, 2: 16, 3: 3}
  Test:  {0: 15, 1: 4, 2: 14, 3: 5}


### F.4 Model Technical Performance

In [48]:
train_sil = silhouette_score(X_train, train_labels)
val_sil   = silhouette_score(X_val, val_labels)
train_db  = davies_bouldin_score(X_train, train_labels)
val_db    = davies_bouldin_score(X_val, val_labels)
train_ch  = calinski_harabasz_score(X_train, train_labels)
val_ch    = calinski_harabasz_score(X_val, val_labels)
print('=== FINAL MODEL PERFORMANCE ===')
print(f'Silhouette Score     - Train: {train_sil:.4f} | Val: {val_sil:.4f}')
print(f'Davies-Bouldin Score - Train: {train_db:.4f} | Val: {val_db:.4f}')
print(f'Calinski-Harabasz    - Train: {train_ch:.2f} | Val: {val_ch:.2f}')
print(f'Inertia              - Train: {kmeans_final.inertia_:.2f}')

=== FINAL MODEL PERFORMANCE ===
Silhouette Score     - Train: 0.3683 | Val: 0.4163
Davies-Bouldin Score - Train: 1.0246 | Val: 0.8919
Calinski-Harabasz    - Train: 91.63 | Val: 21.17
Inertia              - Train: 957.16


In [49]:
centroids_df = pd.DataFrame(kmeans_final.cluster_centers_, columns=X_train.columns)
centroids_df.index.name = 'cluster'
print('Cluster Centroids (standardised):')
centroids_df.round(3)

Cluster Centroids (standardised):


,log_avg_cost,cost_std,cost_change,log_avg_price,price_std,price_change,margin_pct,price_to_cost_ratio,num_offers,avg_discount_pct,max_discount_pct,cost_tier,is_discounted,price_stability_score
cluster,,,,,,,,,,,,,,
0,0.445,-0.361,-0.365,0.401,-0.373,-0.380,-0.706,-0.699,-0.145,-0.303,-0.319,0.372,-0.085,-0.445
1,0.885,1.860,1.886,0.878,1.859,1.901,-0.622,-0.658,-0.135,-0.436,-0.445,0.957,-0.132,1.822
2,-1.160,-0.465,-0.475,-1.109,-0.388,-0.401,1.226,1.281,0.012,-0.391,-0.394,-1.064,-0.251,-0.387
3,0.243,-0.150,-0.148,0.264,-0.286,-0.285,0.051,-0.047,0.560,2.290,2.356,0.154,0.972,-0.035


In [62]:
pca = PCA(n_components=2, random_state=42)
X_pca = pca.fit_transform(X_train)
print("Explained variance ratio:")
print(pca.explained_variance_ratio_)
vis_df = pd.DataFrame({
    'PC1': X_pca[:,0], 'PC2': X_pca[:,1],
    'cluster': train_labels.astype(str)})
alt.Chart(vis_df).mark_point(size=70, opacity=0.8).encode(
    x=alt.X('PC1:Q', title=f'PC1 ({pca.explained_variance_ratio_[0]*100:.1f}% var)'),
    y=alt.Y('PC2:Q', title=f'PC2 ({pca.explained_variance_ratio_[1]*100:.1f}% var)'),
    color=alt.Color('cluster:N', title='Cluster'),
    tooltip=['PC1','PC2','cluster']
).properties(title=f'K-Means (k={OPTIMAL_K}) - PCA Projection', width=500, height=380)

Explained variance ratio:
[0.43038448 0.21513012]


alt.Chart(...)

In [70]:
centroids_pca = pca.transform(kmeans_final.cluster_centers_)

centroid_df = pd.DataFrame({
    'PC1': centroids_pca[:,0],
    'PC2': centroids_pca[:,1]
})

print(centroid_df)

        PC1       PC2
0  0.192838  0.331226
1  4.305180 -1.030553
2 -2.424124 -1.222008
3 -0.310259  3.028694


In [51]:
print('Cluster Business Profiles (from centroids):')
for i in range(OPTIMAL_K):
    c = centroids_df.loc[i]
    print(f'\nCluster {i}:')
    print(f'  Log Avg Price:  {c["log_avg_price"]:.3f}')
    print(f'  Margin %:       {c["margin_pct"]:.3f}')
    print(f'  Avg Discount:   {c["avg_discount_pct"]:.3f}')
    print(f'  Is Discounted:  {c["is_discounted"]:.3f}')

Cluster Business Profiles (from centroids):

Cluster 0:
  Log Avg Price:  0.401
  Margin %:       -0.706
  Avg Discount:   -0.303
  Is Discounted:  -0.085

Cluster 1:
  Log Avg Price:  0.878
  Margin %:       -0.622
  Avg Discount:   -0.436
  Is Discounted:  -0.132

Cluster 2:
  Log Avg Price:  -1.109
  Margin %:       1.226
  Avg Discount:   -0.391
  Is Discounted:  -0.251

Cluster 3:
  Log Avg Price:  0.264
  Margin %:       0.051
  Avg Discount:   2.290
  Is Discounted:  0.972


In [71]:
model_performance_explanations = """
Final K-Means Model Performance Analysis:

Silhouette Score: The validation silhouette score above 0.4 indicates reasonably strong separation between product clusters. For real-world retail datasets with overlapping product behaviour, this represents meaningful and actionable segmentation quality.Final model outperforms baseline (k=2) in granularity and silhouette score.
Small train/val difference is expected and acceptable for unsupervised learning.

Davies-Bouldin: Lower score confirms clusters are well-separated and internally compact.
Calinski-Harabasz: Higher values indicate denser and better-separated clusters.

Cluster Interpretation (example from centroid analysis):
Cluster 0: Standard Products - Moderate pricing with relatively low margins and limited discount activity.

Cluster 1: High-Cost Premium Products - Higher prices and costs with stable pricing behaviour and minimal discounting.

Cluster 2: Low-Cost High-Margin Products - Lower-priced products generating relatively strong margins with low promotional activity.

Cluster 3: Promotional/Discounted Products - Products heavily associated with discounts and promotional campaigns.
PCA scatter plot shows visually distinct groupings confirming cluster separation.

Cluster centroid positions in PCA space help visualise the relative separation and similarity between product groups. Larger distances between centroids indicate stronger segmentation boundaries.

"""

In [72]:
# Do not modify
print_tile(size='h3', key='model_performance_explanations', value=model_performance_explanations)

### F.5 Business Impact from Current Model Performance

In [54]:
print('Products per cluster (training):')
print(pd.Series(train_labels).value_counts().sort_index())
print('Total products modelled:',
      len(train_labels)+len(val_labels)+len(test_labels))

Products per cluster (training):
0    70
1    29
2    54
3    24
Name: count, dtype: int64
Total products modelled: 253


In [55]:
business_impacts_explanations = """
POSITIVE IMPACTS:
1. PRICING STRATEGY: Differentiated pricing rules per cluster.
   Premium cluster maintains full price; clearance cluster can be aggressively discounted.
2. PROMOTIONAL EFFICIENCY: Direct marketing spend at the right product segments.
3. MARGIN PROTECTION: Reduce unnecessary discounting on high-margin products.
4. PRODUCT LIFECYCLE: Low-margin and high-discount products are discontinuation candidates.

RISKS:
- False Premium Classification: under-investment in promotions loses volume sales
- False Clearance Classification: unnecessary markdowns erode brand and margin
- Cluster Drift: prices and costs change over time so quarterly retraining is needed
"""

In [56]:
# Do not modify
print_tile(size='h3', key='business_impacts_explanations', value=business_impacts_explanations)

In [65]:
ethical_considerations = """
ETHICAL CONSIDERATIONS:

The clustering model supports business decision-making but should not fully automate pricing or promotional decisions without human oversight.

Incorrect product segmentation could lead to unfair pricing strategies, excessive discounting, or reduced visibility for certain product categories.

The organisation should regularly review clustering outputs to ensure fairness, transparency, and alignment with customer expectations. Data privacy and responsible use of customer-related business data should also be maintained throughout future model deployment.
"""

In [66]:
print_tile(size='h3', key='ethical_considerations', value=ethical_considerations)

## G. Project Outcomes

In [57]:
experiment_outcome = "Hypothesis Confirmed"

In [58]:
# Do not modify this code
print_tile(size='h2', key='experiment_outcome', value=experiment_outcome)

In [67]:
experiment_results_explanations = """
OUTCOME: Hypothesis Confirmed
Products naturally cluster into distinct segments based on price, cost, margin, and discount behaviour.

KEY INSIGHTS:
1. Product portfolio has distinct natural tiers confirmed by data not just assumption
2. Discount exposure is a strong differentiator - some products are chronically over-discounted
3. Price and cost are correlated but margin varies significantly within price tiers
4. 14 engineered features provided sufficient signal without manual labelling

MODEL COMPARISON:
  Baseline (k=2): Silhouette around 0.35 - crude two-tier separation only
  Final K-Means (optimal k): Higher silhouette - actionable business segments identified
  DBSCAN: Too many noise points for this structured dataset - not recommended

NEXT STEPS:
1. Incorporate sales volume to weight clusters by revenue contribution (HIGH PRIORITY)
2. Retrain model quarterly as price and cost data updates (MEDIUM PRIORITY)
3. Test Gaussian Mixture Models for soft cluster assignment (MEDIUM PRIORITY)

DEPLOYMENT RECOMMENDATION:
Deploy K-Means as a quarterly batch job re-clustering all products.
Output cluster labels to BI dashboard for sales and pricing teams.
Alert when a product changes cluster between quarters as a priority commercial review item.
The model demonstrates strong practical value because clustering can be integrated into pricing systems, promotional planning, and inventory management workflows with relatively low implementation complexity.
"""

In [68]:
# Do not modify
print_tile(size='h2', key='experiment_results_explanations', value=experiment_results_explanations)